# Activation Space vs. Behavior Space: A Goodfire Demo

> **Educational miniature** of the Goodfire Manifold Steering approach (arXiv:2605.05115).
> Not a full paper reproduction — an illustration of the core geometric claim on GPT-2.

**Core claim (Goodfire):** The geometry of how a model *internally represents* concepts
(activation space) approximately mirrors the geometry of how it *behaves* on those concepts
(behavior space). This approximate match is called an **isometry**.

**Model:** GPT-2 (local, ~500 MB, no API key)
**Concepts:** Days of the week — an ordered cyclic set with clear sequential structure in
GPT-2's training data. This matches the example used in the Goodfire paper.

In [ ]:
import warnings
from itertools import combinations
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from scipy.stats import pearsonr
from scipy.spatial.distance import euclidean
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

from prompts import CONCEPT_PROMPTS, DAYS

print(f"Concepts: {DAYS}")
print(f"Total prompts: {sum(len(v) for v in CONCEPT_PROMPTS.values())}")

## Section 1: What Is Approximate Isometry? (Synthetic)

Before loading any model, we understand "approximate isometry" geometrically.

Synthetic activation and behavior centroids are arranged in a cycle (like days of the week),
related by a noisy linear map. The isometry check: do their pairwise distances correlate?

If the scatter of (d_activation, d_behavior) for all concept pairs falls near a line,
the two spaces are approximately isometric — same shape, possibly different scale/rotation.

In [ ]:
np.random.seed(42)
N = 7
short_labels = [d[:3] for d in DAYS]
angles    = np.linspace(0, 2 * np.pi, N, endpoint=False)
act_synth = np.column_stack([np.cos(angles), np.sin(angles)]) * 2.0

A         = np.array([[0.8, -0.5], [0.3, 1.2]])
beh_synth = (A @ act_synth.T).T + np.random.randn(N, 2) * 0.18

colors_synth = [plt.cm.tab10(i / 7) for i in range(N)]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, data, title in [
    (axes[0], act_synth, "Synthetic Activation Space"),
    (axes[1], beh_synth, "Synthetic Behavior Space"),
]:
    for i in range(N):
        ax.scatter(*data[i], color=colors_synth[i], s=130, zorder=3)
        ax.annotate(short_labels[i], data[i] + np.array([0.08, 0.08]), fontsize=10)
    for i in range(N):
        j = (i + 1) % N
        ax.plot([data[i,0], data[j,0]], [data[i,1], data[j,1]],
                color="gray", alpha=0.4, lw=1.5)
    ax.set_title(title, fontsize=12)
    ax.set_aspect("equal")
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")

plt.suptitle("Same cyclic concept geometry — two different spaces", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
pair_idxs   = list(combinations(range(N), 2))
d_act_synth = np.array([np.linalg.norm(act_synth[i]-act_synth[j]) for i,j in pair_idxs])
d_beh_synth = np.array([np.linalg.norm(beh_synth[i]-beh_synth[j]) for i,j in pair_idxs])
adj_synth   = np.array([abs(i-j)==1 or abs(i-j)==N-1 for i,j in pair_idxs])

r_synth, _ = pearsonr(d_act_synth, d_beh_synth)
m_synth     = np.polyfit(d_act_synth, d_beh_synth, 1)
x_line      = np.linspace(d_act_synth.min(), d_act_synth.max(), 100)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(d_act_synth[adj_synth],  d_beh_synth[adj_synth],
           color="steelblue", s=80, alpha=0.9, label="Adjacent pairs")
ax.scatter(d_act_synth[~adj_synth], d_beh_synth[~adj_synth],
           color="coral",     s=60, alpha=0.7, label="Non-adjacent pairs")
ax.plot(x_line, np.polyval(m_synth, x_line), "k--", lw=1.5,
        label=f"fit (slope={m_synth[0]:.2f})")
ax.set_xlabel("d_activation (synthetic)", fontsize=11)
ax.set_ylabel("d_behavior (synthetic)",   fontsize=11)
ax.set_title(f"Pairwise Distance Scatter\nPearson r = {r_synth:.3f}", fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Synthetic Pearson r = {r_synth:.4f}  (points near a line: isometry by construction)")